In [0]:
from pyspark.sql import functions as F

def clean_patient():
    df = spark.table("silver.patient").filter("is_current = true")
    cleaned = df.select(
        "resource_id",
        F.col("resource_json.gender").alias("gender"),
        F.col("resource_json.birthDate").alias("birth_date"),
        F.expr("resource_json.name[0].family").alias("family_name"),
        F.expr("resource_json.name[0].given[0]").alias("given_name"),
        "valid_from", "valid_to", "is_current",
    ).dropDuplicates(["resource_id"])
    cleaned.write.format("delta").mode("overwrite").saveAsTable("silver.patient_clean")
    print(f"patient_clean: {cleaned.count()} rows")

def clean_encounter():
    df = spark.table("silver.encounter").filter("is_current = true")
    cleaned = df.select(
        "resource_id",
        F.col("resource_json.status").alias("status"),
        F.regexp_extract("resource_json.subject.reference", "Patient/(.*)", 1).alias("patient_id"),
        F.col("resource_json.period.start").alias("period_start"),
        F.col("resource_json.period.end").alias("period_end"),
        "valid_from", "valid_to", "is_current",
    ).dropDuplicates(["resource_id"])
    cleaned.write.format("delta").mode("overwrite").saveAsTable("silver.encounter_clean")
    print(f"encounter_clean: {cleaned.count()} rows")

def clean_observation():
    df = spark.table("silver.observation").filter("is_current = true")
    cleaned = df.select(
        "resource_id",
        F.regexp_extract("resource_json.subject.reference", "Patient/(.*)", 1).alias("patient_id"),
        F.regexp_extract("resource_json.encounter.reference", "Encounter/(.*)", 1).alias("encounter_id"),
        F.expr("resource_json.code.coding[0].code").alias("code"),
        F.expr("resource_json.code.coding[0].display").alias("code_display"),
        F.col("resource_json.valueQuantity.value").alias("value_numeric"),
        F.col("resource_json.valueQuantity.unit").alias("value_unit"),
        F.col("resource_json.valueString").alias("value_string"),
        "valid_from", "valid_to", "is_current",
    ).dropDuplicates(["resource_id"])
    cleaned.write.format("delta").mode("overwrite").saveAsTable("silver.observation_clean")
    print(f"observation_clean: {cleaned.count()} rows")

def clean_condition():
    df = spark.table("silver.condition").filter("is_current = true")
    cleaned = df.select(
        "resource_id",
        F.regexp_extract("resource_json.subject.reference", "Patient/(.*)", 1).alias("patient_id"),
        F.regexp_extract("resource_json.encounter.reference", "Encounter/(.*)", 1).alias("encounter_id"),
        F.expr("resource_json.code.coding[0].code").alias("condition_code"),
        F.expr("resource_json.code.coding[0].display").alias("condition_display"),
        F.expr("resource_json.clinicalStatus.coding[0].code").alias("clinical_status"),
        F.col("resource_json.onsetDateTime").alias("onset_date"),
        "valid_from", "valid_to", "is_current",
    ).dropDuplicates(["resource_id"])
    cleaned.write.format("delta").mode("overwrite").saveAsTable("silver.condition_clean")
    print(f"condition_clean: {cleaned.count()} rows")

clean_patient()
clean_encounter()
clean_observation()
clean_condition()

In [0]:
spark.table('silver.encounter_clean').select('resource_id', 'patient_id', 'status').show(5) 